# Alfido Tech Website Traffic Analysis - User Journey, Bounce Rate & Conversion Insights
### InterSpark Internship - Task 3

**Goal:** Analyze website traffic logs to understand user journeys, top landing pages, bounce rates and referral sources.

**Dataset:** 226,278 traffic events | 3,839 unique links | Aug 19-25, 2021

**Tools:** Python, Pandas, Matplotlib, Seaborn, FPDF | Google Colab

---

In [ ]:
# Cell 1 - Install and import
!pip install -q fpdf2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('All libraries ready!')

In [ ]:
# Cell 2 - Upload file
from google.colab import files
uploaded = files.upload()

In [ ]:
# Cell 3 - Unzip if needed
import zipfile, os
zip_files = [f for f in os.listdir('/content') if f.endswith('.zip')]
if zip_files:
    with zipfile.ZipFile(f'/content/{zip_files[0]}', 'r') as z:
        z.extractall('/content/')
        print('Unzipped!')
csv_files = [f for f in os.listdir('/content') if f.endswith('.csv')]
print('CSV files:', csv_files)

In [ ]:
# Cell 4 - Load data
filename = [f for f in os.listdir('/content') if f.endswith('.csv')][0]
df = pd.read_csv(f'/content/{filename}')
print('Loaded:', filename)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Event types:')
print(df['event'].value_counts())
print('Missing values:')
print(df.isnull().sum())
df.head()

In [ ]:
# Cell 5 - Clean data
df['date']     = pd.to_datetime(df['date'])
df['country']  = df['country'].fillna('Unknown')
df['city']     = df['city'].fillna('Unknown')
df['artist']   = df['artist'].fillna('Unknown')
df['album']    = df['album'].fillna('Unknown')
df['track']    = df['track'].fillna('Unknown')
df['hour']     = df['date'].dt.hour
df['day']      = df['date'].dt.date
df['day_name'] = df['date'].dt.day_name()
print('Data cleaned!')
print('Date range:', df['date'].min(), 'to', df['date'].max())
df.head()

In [ ]:
# Cell 6 - Compute metrics
total_events    = len(df)
total_pageviews = len(df[df['event'] == 'pageview'])
total_clicks    = len(df[df['event'] == 'click'])
total_previews  = len(df[df['event'] == 'preview'])
total_links     = df['linkid'].nunique()
total_artists   = df['artist'].nunique()
total_countries = df['country'].nunique()
ctr             = (total_clicks / total_pageviews) * 100
preview_rate    = (total_previews / total_pageviews) * 100
link_events     = df.groupby('linkid')['event'].count()
link_clicks     = df[df['event'] == 'click'].groupby('linkid')['event'].count()
bounced_links   = link_events[(link_events == 1) & (~link_events.index.isin(link_clicks.index))]
bounce_rate     = (len(bounced_links) / total_links) * 100
print('=' * 45)
print('      ALFIDO TECH - TRAFFIC METRICS')
print('=' * 45)
print(f'  Total Events       : {total_events:,}')
print(f'  Total Pageviews    : {total_pageviews:,}')
print(f'  Total Clicks       : {total_clicks:,}')
print(f'  Total Previews     : {total_previews:,}')
print(f'  Unique Links       : {total_links:,}')
print(f'  Click-through Rate : {ctr:.1f}%')
print(f'  Preview Rate       : {preview_rate:.1f}%')
print(f'  Bounce Rate        : {bounce_rate:.1f}%')
print('=' * 45)

In [ ]:
# Cell 7 - Chart 1: Daily trend
sns.set_style('darkgrid')
daily = df.groupby(['day', 'event']).size().reset_index(name='count')
plt.figure(figsize=(14, 5))
for event, color in zip(['pageview','click','preview'],['#6C5CE7','#00B894','#E17055']):
    data = daily[daily['event'] == event]
    plt.plot(data['day'], data['count'], marker='o', linewidth=2.5, label=event.capitalize(), color=color)
plt.title('Daily Traffic Trend (Aug 19-25, 2021)', fontsize=16)
plt.xlabel('Date'); plt.ylabel('Number of Events'); plt.legend(); plt.xticks(rotation=15)
plt.tight_layout(); plt.savefig('chart1_daily_trend.png', dpi=150); plt.show()
print('Chart 1 done!')

In [ ]:
# Cell 8 - Chart 2: Event breakdown
event_counts = df['event'].value_counts()
plt.figure(figsize=(8, 5))
plt.bar(event_counts.index, event_counts.values, color=['#6C5CE7','#00B894','#E17055'])
plt.title('Event Type Breakdown', fontsize=16)
plt.xlabel('Event Type'); plt.ylabel('Count')
for i, v in enumerate(event_counts.values): plt.text(i, v+500, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout(); plt.savefig('chart2_events.png', dpi=150); plt.show()
print('Chart 2 done!')

In [ ]:
# Cell 9 - Chart 3: Top countries
top_countries = df[df['country'] != 'Unknown']['country'].value_counts().head(10)
plt.figure(figsize=(10, 6))
plt.barh(top_countries.index[::-1], top_countries.values[::-1], color='#6C5CE7')
plt.title('Top 10 Countries by Traffic', fontsize=16); plt.xlabel('Number of Events')
for i, v in enumerate(top_countries.values[::-1]): plt.text(v+200, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('chart3_countries.png', dpi=150); plt.show()
print('Chart 3 done!')

In [ ]:
# Cell 10 - Chart 4: Top artists
top_artists = df[df['artist'] != 'Unknown']['artist'].value_counts().head(10)
plt.figure(figsize=(10, 6))
plt.barh(top_artists.index[::-1], top_artists.values[::-1], color='#00B894')
plt.title('Top 10 Artists by Traffic', fontsize=16); plt.xlabel('Number of Events')
for i, v in enumerate(top_artists.values[::-1]): plt.text(v+200, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('chart4_artists.png', dpi=150); plt.show()
print('Chart 4 done!')

In [ ]:
# Cell 11 - Chart 5: Hourly heatmap
heatmap_data = df.groupby(['day_name','hour']).size().unstack(fill_value=0)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = heatmap_data.reindex([d for d in day_order if d in heatmap_data.index])
plt.figure(figsize=(16, 5))
sns.heatmap(heatmap_data, cmap='Purples', linewidths=0.3, cbar_kws={'label':'Number of Events'})
plt.title('Hourly Traffic Heatmap', fontsize=16); plt.xlabel('Hour of Day'); plt.ylabel('Day of Week')
plt.tight_layout(); plt.savefig('chart5_heatmap.png', dpi=150); plt.show()
print('Chart 5 done!')

In [ ]:
# Cell 12 - Chart 6: Funnel
funnel_df = pd.DataFrame({'Stage':['Pageview','Preview','Click'],'Count':[total_pageviews,total_previews,total_clicks]})
plt.figure(figsize=(8, 5))
plt.barh(funnel_df['Stage'][::-1], funnel_df['Count'][::-1], color=['#6C5CE7','#FDCB6E','#00B894'])
plt.title('User Flow Funnel - Pageview to Click', fontsize=16); plt.xlabel('Number of Events')
for i, v in enumerate(funnel_df['Count'][::-1]):
    pct = (v / total_pageviews) * 100
    plt.text(v+500, i, f'{v:,} ({pct:.1f}%)', va='center', fontsize=10)
plt.tight_layout(); plt.savefig('chart6_funnel.png', dpi=150); plt.show()
print('Chart 6 done!')

In [ ]:
# Cell 13 - Chart 7: Top landing pages
top_links = df[df['event'] == 'pageview']['linkid'].value_counts().head(10)
plt.figure(figsize=(10, 6))
plt.barh(range(len(top_links)), top_links.values[::-1], color='#E17055')
plt.yticks(range(len(top_links)), [f'Link {i+1}' for i in range(len(top_links))][::-1])
plt.title('Top 10 Landing Pages by Pageviews', fontsize=16); plt.xlabel('Number of Pageviews')
for i, v in enumerate(top_links.values[::-1]): plt.text(v+50, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('chart7_landing_pages.png', dpi=150); plt.show()
print('Chart 7 done!')

In [ ]:
# Cell 14 - Generate PDF and download
from fpdf import FPDF
import datetime

class PDF(FPDF):
    def header(self):
        self.set_fill_color(108, 92, 231); self.rect(0, 0, 210, 25, 'F')
        self.set_text_color(255,255,255); self.set_font('Helvetica','B',16)
        self.set_xy(0,7); self.cell(0,10,'Alfido Tech - Website Traffic Analysis Report',align='C')
        self.set_text_color(0,0,0)
    def footer(self):
        self.set_y(-15); self.set_font('Helvetica','I',8); self.set_text_color(150,150,150)
        self.cell(0,10,f'Page {self.page_no()} | Generated {datetime.date.today()}',align='C')

pdf = PDF()
pdf.set_auto_page_break(auto=True, margin=20)
pdf.add_page(); pdf.ln(20)
pdf.set_font('Helvetica','B',20); pdf.set_text_color(108,92,231)
pdf.cell(0,10,'Executive Summary',ln=True); pdf.ln(3)
pdf.set_font('Helvetica','',11); pdf.set_text_color(50,50,50)
pdf.multi_cell(0,7,'This report analyzes website traffic logs for Alfido Tech covering 226,278 events across 7 days (August 19-25, 2021). Five optimization recommendations are provided to improve conversions.')
pdf.ln(5)
pdf.set_font('Helvetica','B',13); pdf.set_text_color(108,92,231)
pdf.cell(0,8,'Key Metrics',ln=True); pdf.ln(2)
for label, value in [('Total Events',f'{total_events:,}'),('Pageviews',f'{total_pageviews:,}'),('Clicks',f'{total_clicks:,}'),('Previews',f'{total_previews:,}'),('Unique Links',f'{total_links:,}'),('CTR',f'{ctr:.1f}%'),('Preview Rate',f'{preview_rate:.1f}%'),('Bounce Rate',f'{bounce_rate:.1f}%')]:
    pdf.set_fill_color(240,238,255); pdf.set_text_color(60,60,60)
    pdf.cell(80,9,f'  {label}',border=0,fill=True)
    pdf.set_font('Helvetica','B',11); pdf.cell(110,9,value,border=0,fill=False,ln=True)
    pdf.set_font('Helvetica','',11); pdf.ln(1)
for title, img in [('Traffic Trends','chart1_daily_trend.png'),('Hourly Heatmap','chart5_heatmap.png'),('Event Breakdown','chart2_events.png'),('User Flow Funnel','chart6_funnel.png'),('Top Countries','chart3_countries.png'),('Top Artists','chart4_artists.png'),('Top Landing Pages','chart7_landing_pages.png')]:
    pdf.add_page(); pdf.ln(20)
    pdf.set_font('Helvetica','B',18); pdf.set_text_color(108,92,231)
    pdf.cell(0,10,title,ln=True); pdf.ln(4)
    pdf.image(img,x=15,w=178)
pdf.add_page(); pdf.ln(20)
pdf.set_font('Helvetica','B',18); pdf.set_text_color(108,92,231)
pdf.cell(0,10,'5 Optimization Recommendations',ln=True); pdf.ln(4)
for title, body in [
    ('1. Optimize top landing pages','Top 10 links drive most traffic. Add clear CTAs, improve load speed and A/B test layouts to boost click-through rates.'),
    ('2. Target Saudi Arabia and India','Create localized pages in Arabic and Hindi and run geo-targeted campaigns to convert this high-volume audience.'),
    ('3. Improve preview-to-click conversion','Add autoplay previews and show streaming platform options prominently to push more users from preview to click.'),
    ('4. Capitalize on peak traffic hours','Schedule social media posts and campaigns to go live just before peak hours to maximize visibility.'),
    ('5. Leverage Tesher dominance','Use Tesher pages as a template, cross-promote similar artists and study what makes these pages convert so well.'),
]:
    pdf.set_fill_color(108,92,231); pdf.set_text_color(255,255,255)
    pdf.set_font('Helvetica','B',12); pdf.cell(0,9,f'  {title}',fill=True,ln=True)
    pdf.set_fill_color(245,243,255); pdf.set_text_color(50,50,50)
    pdf.set_font('Helvetica','',10); pdf.multi_cell(0,6,f'  {body}',fill=True); pdf.ln(3)
pdf.output('alfido_tech_traffic_report.pdf')
print('PDF created!')
from google.colab import files
files.download('alfido_tech_traffic_report.pdf')
print('Downloaded!')